# Day 068 — Exercise 2: Preprocess for OCR

**What you'll build:** `preprocess_for_ocr(img) -> Image.Image` — three preprocessing steps that significantly improve Tesseract accuracy.

**Why it matters:** Tesseract's LSTM recogniser needs high-contrast, sufficiently-sized characters. These three steps (grayscale → contrast → scale) are the most impactful preprocessing operations — often raising character accuracy from 60% to 98% on real-world document scans.

In [ ]:
from PIL import Image, ImageEnhance

_small_rgb  = Image.new('RGB',  (200, 80),  color=(180, 180, 200))   # small, grey
_large_rgb  = Image.new('RGB',  (1200, 400), color=(220, 210, 190))  # already large


## Task

Implement `preprocess_for_ocr(img) -> Image.Image`:

1. `out = img.convert('L')` — grayscale
2. `out = ImageEnhance.Contrast(out).enhance(2.0)` — boost contrast
3. If `out.size[0] < 1000`: upscale to 1000 px wide with `Image.Resampling.LANCZOS`
4. Return `out`

Pure PIL — no Tesseract, no mocks needed.

## Your Implementation

In [ ]:
def preprocess_for_ocr(img) -> 'Image.Image':
    """Improve image quality for Tesseract OCR.

    Steps (in order):
        1. Convert to greyscale ('L' mode)
        2. Boost contrast by factor 2.0 using ImageEnhance.Contrast
        3. Upscale to at least 1000 px wide using LANCZOS if narrower

    Args:
        img: PIL Image (any mode)
    Returns:
        Preprocessed PIL Image in 'L' mode
    """
    raise NotImplementedError


In [ ]:
def preprocess_for_ocr(img):
    out = img.convert('L')
    out = ImageEnhance.Contrast(out).enhance(2.0)
    w, h = out.size
    if w < 1000:
        scale = 1000 / w
        out = out.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return out


## Automated checks

In [ ]:
score, total = 0, 5
try:
    result = preprocess_for_ocr(_small_rgb)
    assert hasattr(result, 'mode'), "Expected a PIL Image"
    score += 1; print("\u2705 returns a PIL Image")

    assert result.mode == 'L', f"Expected 'L' mode, got {result.mode!r}"
    score += 1; print("\u2705 output is grayscale ('L' mode)")

    w, h = result.size
    assert w >= 1000, f"Expected width >= 1000 px, got {w}"
    score += 1; print(f"\u2705 small image upscaled to {w}x{h}")

    # Already-large image should not shrink
    large_result = preprocess_for_ocr(_large_rgb)
    lw, lh = large_result.size
    assert lw >= 1200, f"Large image should not shrink: got {lw}"
    score += 1; print(f"\u2705 large image not shrunk (width {lw})")

    # RGBA input also works
    rgba = Image.new('RGBA', (300, 100), color=(200, 50, 50, 128))
    preproc = preprocess_for_ocr(rgba)
    assert preproc.mode == 'L'
    score += 1; print("\u2705 RGBA input converted to L without error")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def preprocess_for_ocr(img):
    out = img.convert('L')
    out = ImageEnhance.Contrast(out).enhance(2.0)
    w, h = out.size
    if w < 1000:
        scale = 1000 / w
        out = out.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
    return out
```

**Why check `w < 1000` not `w < 1000 or h < 1000`?** Width is the bottleneck because text lines run horizontally — a narrow image has narrow characters. Height scales with width proportionally so checking only width avoids distorting aspect ratios of tall narrow images.

</details>